# Linear Regression using Topics

We're going to run a simple linear regression trying to discover if some features affect the CTR of the headlines when we look at specific topics.

In [12]:
import pandas as pd

exploratory_df = pd.read_csv("dataset/processed/exploratory-packages-topic-modeling.csv")
confirmatory_df = pd.read_csv("dataset/processed/confirmatory-packages-topic-modeling.csv")
holdout_df = pd.read_csv("dataset/processed/holdout-packages-topic-modeling.csv")

exploratory_df.head()

,package_id,created_at,test_week,clickability_test_id,headline,eyecatcher_id,impressions,clicks,ctr,ctr_demeaned,...,neu,pos,compound,created_at_dayofweek,created_at_hourofday,test_group_size,read_flesch,read_coleman,specificity_tfidf,topic_bertopic
0,62,2014-11-20 14:57:52.478,201446,546e009a9ad54ec65b00004b,What They Learned From The Scientist Was Terri...,546c7f2dbadeb5788700000a,4594,51,0.011101,0.111516,...,0.856,0.000,-0.3291,3,14,6,80.782500,8.093333,0.444616,6
1,84,2014-11-20 14:54:18.780,201446,546e009a9ad54ec65b00004b,A Science Guy Helps 3 Dudes From America Under...,546c7f2dbadeb5788700000a,4571,58,0.012689,0.682108,...,0.809,0.191,0.3818,3,14,6,59.682143,10.257143,0.370851,6
2,95,2014-11-20 15:04:49.517,201446,546e009a9ad54ec65b00004b,He Sat Them Down And Told Them About An Immine...,546c7f2dbadeb5788700000a,4601,27,0.005868,-1.769714,...,0.813,0.000,-0.5994,3,15,6,80.465000,6.077778,0.398801,-1
3,100,2014-11-20 15:13:36.266,201446,546e009a9ad54ec65b00004b,"The 3 Of Them Needed To See It In Person, And ...",546c7f2dbadeb5788700000a,4567,63,0.013795,1.079669,...,0.720,0.156,0.2023,3,15,6,80.777143,3.228571,0.445514,-1
4,102,2014-11-20 15:15:25.697,201446,546e009a9ad54ec65b00004b,"They May Not Be The Most Handsome Dudes, But T...",546c7f2dbadeb5788700000a,4524,44,0.009726,-0.382964,...,0.655,0.345,0.7812,3,15,6,92.965000,5.875000,0.497006,-1


In [13]:
def prepare_data(df):
    # Transform int columns to float
    df["headline_num_persons"] = df["headline_num_persons"].astype(float)
    df["headline_num_orgs"] = df["headline_num_orgs"].astype(float)
    df["headline_num_gpes"] = df["headline_num_gpes"].astype(float)
    df["num_pronouns"] = df["num_pronouns"].astype(float)
    df["num_chars"] = df["num_chars"].astype(float)
    df["num_tokens"] = df["num_tokens"].astype(float)
    df["num_nouns"] = df["num_nouns"].astype(float)
    df["num_verbs"] = df["num_verbs"].astype(float)
    df["num_adjs"] = df["num_adjs"].astype(float)
    df["num_advs"] = df["num_advs"].astype(float)
    df["test_group_size"] = df["test_group_size"].astype(float)

    # Drop rows with missing values in any column
    df = df.dropna()
    return df

exploratory_df = prepare_data(exploratory_df)
confirmatory_df = prepare_data(confirmatory_df)
holdout_df = prepare_data(holdout_df)


In [14]:
exploratory_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11996 entries, 0 to 12009
Data columns (total 51 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   package_id                     11996 non-null  int64  
 1   created_at                     11996 non-null  object 
 2   test_week                      11996 non-null  int64  
 3   clickability_test_id           11996 non-null  object 
 4   headline                       11996 non-null  object 
 5   eyecatcher_id                  11996 non-null  object 
 6   impressions                    11996 non-null  int64  
 7   clicks                         11996 non-null  int64  
 8   ctr                            11996 non-null  float64
 9   ctr_demeaned                   11996 non-null  float64
 10  first_place                    11996 non-null  bool   
 11  winner                         11996 non-null  bool   
 12  is_highest_ctr                 11996 non-null  bool

In [15]:
import statsmodels.api as sm

def basic_linear_regression(df):
    target_col = "ctr_demeaned"
    feature_cols = [
        "headline_num_persons",
        "headline_num_orgs",
        "headline_num_gpes",
        "headline_has_person",
        "headline_has_org",
        "headline_has_gpe",
        "headline_has_money",
        "starts_with_verb",
        "starts_with_pronoun",
        "starts_with_number",
        "num_pronouns",
        "num_chars",
        "num_tokens",
        "avg_token_len",
        "ends_with_qmark",
        "ends_with_exclaim",
        "has_quote",
        "has_all_caps_word",
        "num_nouns",
        "num_verbs",
        "num_adjs",
        "num_advs",
        "headline_imperative_verb",
        "headline_has_curiosity_word",
        "headline_curiosity_similarity",
        "headline_has_intensity_word",
        "headline_intensity_similarity",
        "neg",
        "neu",
        "pos",
        "compound",
        "test_group_size",
        "read_flesch",
        "read_coleman",
        "specificity_tfidf",
    ]

    # Drop rows with missing values in any of these columns
    model_df = df[feature_cols + [target_col]].dropna()

    X = model_df[feature_cols]
    y = model_df[target_col]

    # Ensure all predictors and target are numeric (convert bools to 0/1 and ints to float)
    X = X.astype(float)
    y = y.astype(float)

    # Add intercept term
    X = sm.add_constant(X)

    # Fit OLS regression using Statsmodels
    return sm.OLS(y, X).fit()

topic_col = "topic_bertopic"
df = exploratory_df.copy()
# Only keep to most important 6 topics, the others ones are the rest
df[topic_col] = df[topic_col].apply(lambda t: t if t in [0,1,2,3,4,5] else -1)

for topic_id in range(-1, 6):
    print(f"Running regression for topic {topic_id}")
    df = exploratory_df[exploratory_df[topic_col] == topic_id].copy()
    print(f"Number of rows: {len(df)}")
    print("=========================================")
    ols_model = basic_linear_regression(df)
    print(ols_model.summary())

Running regression for topic -1
Number of rows: 6668
                            OLS Regression Results                            
Dep. Variable:           ctr_demeaned   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     3.603
Date:                Mon, 15 Dec 2025   Prob (F-statistic):           4.43e-12
Time:                        18:29:13   Log-Likelihood:                -8602.4
No. Observations:                6668   AIC:                         1.728e+04
Df Residuals:                    6632   BIC:                         1.752e+04
Df Model:                          35                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------

It could be due to the reduced number of headlines per topic, but doing this analysis doesn't provide any meaningful results.